# Complaint severity scoring

Turns `sample_complaints.csv` into a scored, cleaned crime layer for RiskMap.

**What it does**
1. Scores every complaint **1–5** (5 = worst) with keyword rules
2. Bags them into **violent / intrusion / property**
3. Drops points that don't sit on the NYC street network
4. Filters to `severity >= MIN_SEVERITY` and exports for Supabase

**Severity scale** — ranked by danger to someone *walking past*, which is what the router cares about:

| # | Meaning | Offenses matched here |
|---|---------|----------------------|
| **5** | Force or threat against a person | murder, rape, shooting, felony assault, **robbery** |
| **4** | Violence or weapons, lesser degree | weapons, **assault 3**, harassment, arson |
| **3** | Intrusion into occupied space | **burglary**, trespass |
| **2** | Property loss or damage | **grand larceny**, **criminal mischief** |
| **1** | Petty property | **petit larceny**, fraud, forgery |

The rules cover the full NYPD offense taxonomy, not just the 6 types in this sample — so swapping in the
real *NYPD Complaint Data Historic* file needs no code changes.

In [ ]:
COMPLAINTS   = "sample_complaints.csv"
CRASHES      = "Motor_Vehicle_Collisions_-_Crashes_20260815.csv"  # used as a street-network reference
OUT_PATH     = "complaints_scored.csv"

MIN_SEVERITY = 4      # keep 4 and 5 — the crimes that matter to a pedestrian
ON_STREET_M  = 100    # a complaint further than this from any street is discarded
NYC_BBOX     = dict(lat=(40.48, 40.93), lon=(-74.27, -73.68))

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

df = pd.read_csv(COMPLAINTS, low_memory=False)
print(f"loaded {len(df):,} complaints")
print(df["OFNS_DESC"].value_counts().to_string())

## 1 · Severity rules

First pattern that matches wins, so order matters — `FELONY ASSAULT` must be tested before plain `ASSAULT`.
Matching on keywords rather than exact strings also survives the truncated labels in this file
(`CRIMINAL MISCHIEF & RELATED OF`).

In [ ]:
SEVERITY_RULES = [
    (r"MURDER|HOMICIDE|MANSLAUGHTER",      5, "violent"),
    (r"RAPE|SEX CRIME|SEXUAL ABUSE",       5, "violent"),
    (r"SHOOTING|FELONY ASSAULT",           5, "violent"),
    (r"ROBBERY",                           5, "violent"),
    (r"WEAPON|FIREARM|DANGEROUS WEAPON",   4, "violent"),
    (r"ASSAULT|HARRASSMENT|HARASSMENT",    4, "violent"),
    (r"ARSON|KIDNAP",                      4, "violent"),
    (r"BURGLARY|TRESPASS",                 3, "intrusion"),
    (r"GRAND LARCENY",                     2, "property"),
    (r"CRIMINAL MISCHIEF|VANDALISM",       2, "property"),
    (r"LARCENY|THEFT|FRAUD|FORGERY",       1, "property"),
]
DEFAULT = (1, "other")


def classify(desc):
    """Return (severity, bucket) for an OFNS_DESC string."""
    if not isinstance(desc, str):
        return DEFAULT
    d = desc.upper().strip()
    for pattern, sev, bucket in SEVERITY_RULES:
        if re.search(pattern, d):
            return sev, bucket
    return DEFAULT

In [ ]:
codes = df["OFNS_DESC"].map(classify)
df["severity"] = [c[0] for c in codes]
df["bucket"] = [c[1] for c in codes]

# Audit: anything that fell through to the default is silently scored 1 — surface it.
unmatched = df.loc[df["bucket"] == "other", "OFNS_DESC"].value_counts()
print(f"unmatched offense types: {len(unmatched)}")
if len(unmatched):
    print(unmatched.to_string())

print("\nresulting map:")
print(
    df.groupby(["severity", "bucket"])["OFNS_DESC"]
    .agg(["count", lambda s: ", ".join(sorted(s.unique())[:3])])
    .rename(columns={"<lambda_0>": "examples"})
    .to_string()
)

## 2 · Clean the geography

This file has coordinates that don't all land on the city — about a third sit off the street network
entirely (open water included). The 2.3M crash records make a good street mask: crashes happen on roads,
so any complaint far from every crash is not on a road.

In [ ]:
before = len(df)
df = df[
    df["Latitude"].between(*NYC_BBOX["lat"])
    & df["Longitude"].between(*NYC_BBOX["lon"])
].copy()
print(f"bbox filter : {before:,} -> {len(df):,}  (dropped {before - len(df):,})")

streets = pd.read_csv(CRASHES, engine="pyarrow", usecols=["LATITUDE", "LONGITUDE"])
streets = streets[
    streets.LATITUDE.between(*NYC_BBOX["lat"])
    & streets.LONGITUDE.between(*NYC_BBOX["lon"])
]
tree = cKDTree(np.c_[streets.LATITUDE.values, streets.LONGITUDE.values])
dist, _ = tree.query(np.c_[df.Latitude.values, df.Longitude.values])
df["street_dist_m"] = dist * 111_000

on = df["street_dist_m"] <= ON_STREET_M
print(f"on-street   : {on.sum():,} / {len(df):,}  ({on.mean() * 100:.1f}%)")
df = df[on].copy()

## 3 · Apply the filter

In [ ]:
hot = df[df["severity"] >= MIN_SEVERITY].copy()
print(f"severity >= {MIN_SEVERITY}: {len(hot):,} of {len(df):,} ({len(hot) / len(df) * 100:.1f}%)")
print()
print(hot.groupby(["severity", "bucket"]).size().to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

sv = df["severity"].value_counts().sort_index()
ax[0].bar(sv.index.astype(str), sv.values,
          color=["#94a3b8", "#94a3b8", "#f59e0b", "#ef4444", "#991b1b"][:len(sv)])
ax[0].set_title("complaints by severity (after cleaning)")
ax[0].set_xlabel("severity")

sc = ax[1].scatter(hot["Longitude"], hot["Latitude"], c=hot["severity"],
                   cmap="YlOrRd", s=4, alpha=0.5, vmin=1, vmax=5, linewidths=0)
ax[1].set_aspect(1 / np.cos(np.radians(40.7)))
ax[1].set_title(f"kept: severity >= {MIN_SEVERITY}")
fig.colorbar(sc, ax=ax[1], shrink=0.8, label="severity")

plt.tight_layout()
plt.show()

## 4 · Export for Supabase

```sql
create table crime_points (
  id          bigserial primary key,
  occurred_on date,
  offense     text,
  severity    smallint,   -- 1-5
  bucket      text,       -- violent | intrusion | property
  weight      real,       -- feeds crime_risk on road_edges
  geom        geography(Point, 4326)
);
create index on crime_points using gist (geom);
```

In [ ]:
out = pd.DataFrame({
    "occurred_on": pd.to_datetime(hot["CMPLNT_FR_DT"], format="%m/%d/%Y").dt.date,
    "offense": hot["OFNS_DESC"].str.strip(),
    "severity": hot["severity"],
    "bucket": hot["bucket"],
    "lat": hot["Latitude"].round(6),
    "lon": hot["Longitude"].round(6),
    "weight": hot["severity"].astype(float),
})
out.to_csv(OUT_PATH, index=False)
print(f"wrote {OUT_PATH}  ({len(out):,} rows)")
print()
print(out.head().to_string())
print()
print(out.groupby("bucket")["weight"].agg(["count", "sum", "mean"]).round(2).to_string())

---
**Note on this data.** `sample_complaints.csv` is a synthetic sample: its `BORO_NM` matches the coordinates
only 25.0% of the time (chance for a 4-way guess), every row carries the timestamp `12:00:00`, and the six
offense types are uniformly distributed. The scoring above is correct and the pipeline is production-shaped,
but the *output* is only as meaningful as the input. Drop the real **NYPD Complaint Data Historic** file in
as `COMPLAINTS` — identical column names — and this notebook produces a genuine crime layer unchanged.